In [ ]:
from openai import OpenAI
import gradio as gr
import time


In [ ]:
# OpenAI client
import os
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI()

In [ ]:
# Transcribe audio
def transcribe_audio(audio_file):
    with open(audio_file, "rb") as f:
        transcript = client.audio.transcriptions.create(
            model="gpt-4o-transcribe",
            file=f
        )
    return transcript.text

In [ ]:
# Stream text (helper)
def stream_text(text, delay=0.02):
    output = ""
    for char in text:
        output += char
        time.sleep(delay)
        yield output


In [ ]:
# Main sreaming function
def process_audio(audio):

    if audio is None:
        yield "❌ No audio recorded.", ""
        return

    print("Processing:", audio)

    # Step 1: Transcription
    transcript = transcribe_audio(audio)

    # Step 2: Summary
    summary_response = client.responses.create(
        model="gpt-5-mini",
        input=f"Summarize this transcript:\n\n{transcript}"
    )

    summary = summary_response.output_text

    # Step 3: Streaming
    transcript_stream = stream_text(f"## Transcript\n\n{transcript}")
    summary_stream = stream_text(f"## Summary\n\n{summary}")

    transcript_output = ""
    summary_output = ""

    for chunk in transcript_stream:
        transcript_output = chunk
        yield transcript_output, summary_output

    for chunk in summary_stream:
        summary_output = chunk
        yield transcript_output, summary_output


In [ ]:

# Gradio UI 
with gr.Blocks() as app:
    gr.Markdown("# 🎤 Speak → Auto Transcript → Summary")

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Speak here (auto processes after recording)"
    )

    transcript_md = gr.Markdown()
    summary_md = gr.Markdown()

    # 🔥 AUTO trigger when recording stops
    audio_input.change(
        fn=process_audio,
        inputs=audio_input,
        outputs=[transcript_md, summary_md]
    )

# Enable streaming queue
app.queue()





In [ ]:
# Run
if __name__ == "__main__":
    app.launch(share=True)